# Counterfactual risk analysis on a Morpho MetaMorpho vault

This notebook walks through the core logic of `morpho-vault-counterfactuals` - pulling vault state, running six counterfactual detectors, inspecting evidence dicts, and producing a curator-style summary.

**Author:** Max Gorbuk | github.com/mkzung  
**Repo:** [github.com/mkzung/morpho-vault-counterfactuals](https://github.com/mkzung/morpho-vault-counterfactuals)

The demo uses a pinned-block snapshot fixture (so anyone cloning the repo can re-run end-to-end without an RPC). The same code paths work against a live `fetch_vault_snapshot(address)` call against the Morpho Blue API.

In [ ]:
import sys; sys.path.insert(0, '../src')
from mvcf import load_fixture, run_all_detectors
from mvcf.runner import summarize

snap = load_fixture('steakhouse_usdc_snapshot_demo')
print(f"Vault:         {snap.vault_address}")
print(f"Block:         {snap.block:,}")
print(f"Total assets:  {snap.total_assets / 1e6:,.0f} USDC")
print(f"# markets:     {len(snap.markets)}")
print(f"# borrowers:   {len(snap.borrowers)}")
print(f"HHI:           {snap.hhi:.3f}  (top-1 share = {snap.top1_share:.1%})")

## 1. Per-market state

Each market the vault is exposed to is a `(collateral_asset, loan_asset, oracle, IRM, LLTV)` tuple. Utilization tells you how much of supplied loan-asset is currently borrowed.

In [ ]:
import pandas as pd
rows = []
for m in snap.markets:
    rows.append({
        'market_id_short': m.market_id[:10] + '...',
        'supply_usdc':     m.total_supply_assets / 1e6,
        'borrow_usdc':     m.total_borrow_assets / 1e6,
        'utilization':     m.utilization,
        'lltv':            m.lltv,
        'supply_cap_usdc': m.supply_cap / 1e6,
    })
pd.DataFrame(rows).style.format({
    'supply_usdc': '${:,.0f}', 'borrow_usdc': '${:,.0f}',
    'utilization': '{:.1%}', 'lltv': '{:.0%}', 'supply_cap_usdc': '${:,.0f}',
})

## 2. Run all six detectors with default Re7-style parameters

Each detector returns a `DetectorResult` with a single fractional metric, a human-readable interpretation, and an `evidence` dict the curator can audit.

In [ ]:
results = run_all_detectors(
    snap,
    oracle_drift=-0.10,       # collateral price drifts -10% while oracle stale
    collateral_shock=-0.20,   # instant -20% step shock on collateral
    top_n_exit=1,             # top-1 depositor exits
    util_band=0.92,           # markets above this utilization are flagged
    gas_gwei=30,              # current mainnet gas-price assumption
)
print(summarize(results))

## 3. Sensitivity analysis - Collateral cascade across multiple shock levels

A real curator sweeps the shock parameter rather than picking a single point estimate.

In [ ]:
from mvcf.detectors import CollateralCascade
import matplotlib.pyplot as plt

shocks = [-0.02, -0.05, -0.10, -0.15, -0.20, -0.25, -0.30, -0.40, -0.50]
fractions = [CollateralCascade(shock_pct=s).run(snap).headline_metric for s in shocks]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot([abs(s) * 100 for s in shocks], [f * 100 for f in fractions], marker='o', linewidth=2)
ax.set_xlabel('Collateral shock magnitude (%)')
ax.set_ylabel('Liquidatable debt (% of total debt)')
ax.set_title('Collateral cascade sensitivity - Steakhouse USDC demo fixture')
ax.grid(alpha=0.3)
ax.set_xlim(0, 55); ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()
for s, f in zip(shocks, fractions):
    print(f'  shock {s:+.0%}  ->  {f:.1%} of debt liquidatable')

## 4. Inspect the evidence dict for the OracleFreezeReplay detector

The single headline number is not the full answer. The `evidence` dict lets a curator audit *which* markets contributed to the risk, *how many* positions are at stake, and verify the math.

In [ ]:
import json
ofr = next(r for r in results if r.name == 'OracleFreezeReplay')
print(json.dumps(ofr.evidence, indent=2, default=str))

## 5. What a curator does with this output

For the demo fixture, the headline reads:

- **57% of debt within 5pp of LLTV** - the vault is running hot. A small adverse oracle move would liquidate a majority of positions.
- **73% liquidatable at -20% shock** - concentrated exposure to one or two markets where borrowers are close to their LLTV.
- **57% bad-debt fraction at oracle freeze + 10% drift** - most of the at-risk debt is in markets where the LLTV is already at the upper end of the safe range.
- **1 of 5 markets above 92% utilization** - IRM curves are steep there; depositor withdrawal pressure will compound interest rates.

A curator's action set is small and well-defined:
1. **Lower LLTV** on the hot market(s) to widen the buffer.
2. **Reduce supply cap** on the same markets to cap further inflow.
3. **Reallocate** existing supply via the MetaMorpho `reallocate()` flow to a safer market.
4. **Raise alert thresholds** if the position trends keep deteriorating between snapshots.

Each of these is a single multi-step on-chain transaction sequenced through the MetaMorpho timelock - exactly the curator intervention this framework is the read-side of.